# 06 — Kotekar Ablation Training (KAGGLE GPU)
## WavSent-MTL · Tasks 3.1–3.12

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Configs A, B, C → 30 seeds each (TKAN encoder)
- After A/B/C: compare B vs C mean val accuracy → set BEST_REPR
- Configs D (LSTM), E (GRU), F (TCN) → 30 seeds each using BEST_REPR input
- Saves best-seed val+test predictions per model (for PSO in notebook 08)
- Saves results CSV after every config (crash-safe)

**PREREQUISITE:** config/config.py `best_params` must be updated from notebook 05 output

**Input shapes:**
- Config A (returns + polarity_mean): n_features varies (computed inside notebook)
- Config B (denoised OHLCV + polarity_mean): 6 features
- Config C (denoised technicals + polarity_mean): 8 features (selected_features)
- Configs D/E/F: BEST_REPR features

In [1]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
import sys
import os

repo_path = '/kaggle/working/WavSent-MTL'

if os.path.exists(repo_path):
    subprocess.run(['git', '-C', repo_path, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone',
        'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
        repo_path], check=True)

sys.path.insert(0, repo_path)
from config.config import CONFIG
print(f"Ready. BEST_REPR: {CONFIG['BEST_REPR']}")
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
import gc
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

print()
print('─' * 45)
print('Training Configuration')
print('─' * 45)
print(f"Early stopping patience : {CONFIG['early_stopping_patience']}")
print(f"Early stopping monitor  : {CONFIG['early_stopping_monitor']}")
print(f"Max epochs              : {CONFIG['max_epochs']}")
print(f"LR reduce patience      : {CONFIG['lr_reduce_patience']}")
print(f"LR reduce monitor       : {CONFIG['lr_reduce_monitor']}")
print(f"LR reduce factor        : {CONFIG['lr_reduce_factor']}")
print(f"Loss type               : {CONFIG['loss_type']}")
print('─' * 45)

Cloning into '/kaggle/working/WavSent-MTL'...


Ready. BEST_REPR: denoised_technicals
PyTorch: 2.10.0+cu128
Device:  cuda
GPU:     Tesla T4

─────────────────────────────────────────────
Training Configuration
─────────────────────────────────────────────
Early stopping patience : 35
Early stopping monitor  : val_binary_accuracy
Max epochs              : 150
LR reduce patience      : 10
LR reduce monitor       : val_loss
LR reduce factor        : 0.5
Loss type               : uncertainty
─────────────────────────────────────────────


## Load Kotekar Processed Arrays

In [2]:
import numpy as np
import pandas as pd
import json

KOTEKAR_INPUT = '/kaggle/input/datasets/consistency23/wavsent-kotekar-processed'

# Full data dict (Config C / selected features — 8 features)
data_C = {
    'X_train':      np.load(f'{KOTEKAR_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KOTEKAR_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KOTEKAR_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KOTEKAR_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KOTEKAR_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KOTEKAR_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KOTEKAR_INPUT}/y_reg_test.npy'),
}

with open(f'{KOTEKAR_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)   # 8 features

with open(f'{KOTEKAR_INPUT}/class_weights.json') as f:
    class_weights = json.load(f)       # None or {0: w0, 1: w1}

print(f'X_train shape:     {data_C["X_train"].shape}')
print(f'Selected features: {selected_features}')
print(f'Class weights:     {class_weights}')

X_train shape:     (741, 5, 8)
Selected features: ['WILLIAMS_R', 'STOCH_K', 'MACD', 'RSI_14', 'CCI_20', 'EMA_9', 'Volume_d', 'polarity_mean']
Class weights:     None


## Build Config A and B Data Variants

Config A uses daily returns + polarity_mean (Kotekar-spirit baseline).  
Config B uses denoised OHLCV (5 cols) + polarity_mean = 6 features.  
Both are subsets of featured_data.csv — reconstruct from the full arrays.

In [3]:
# We need featured_data.csv to rebuild Config A and B inputs.
# Upload data/processed/kotekar/featured_data.csv as part of the Kaggle dataset.
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from src.data.windows import create_windows, generate_targets, temporal_split
from src.data.preprocessor import apply_scaler, apply_reg_scaler

df_feat = pd.read_csv(f'{KOTEKAR_INPUT}/featured_data.csv')
df_feat['Date'] = pd.to_datetime(df_feat['Date'])
df_feat = df_feat.sort_values('Date').reset_index(drop=True)

# Temporal split (same split as notebook 04)
train_df, val_df, test_df = temporal_split(df_feat)
print(f'Split: Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')

def build_data_variant(feature_cols, train_df, val_df, test_df, w=5):
    """Scale, window, and target-generate for given feature columns."""
    tr_s, v_s, te_s, _ = apply_scaler(
        train_df[feature_cols].values,
        val_df[feature_cols].values,
        test_df[feature_cols].values,
        save_path='/tmp/tmp_scaler.pkl'
    )
    X_train = create_windows(tr_s, w)
    X_val   = create_windows(v_s, w)
    X_test  = create_windows(te_s, w)
    y_clf_tr, y_reg_tr = generate_targets(train_df['Close'].values, w)
    y_clf_v,  y_reg_v  = generate_targets(val_df['Close'].values, w)
    y_clf_te, y_reg_te = generate_targets(test_df['Close'].values, w)
    y_reg_tr, y_reg_v, y_reg_te, _ = apply_reg_scaler(
        y_reg_tr, y_reg_v, y_reg_te, save_path='/tmp/tmp_reg_scaler.pkl')
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_clf_train': y_clf_tr, 'y_clf_val': y_clf_v, 'y_clf_test': y_clf_te,
        'y_reg_train': y_reg_tr, 'y_reg_val': y_reg_v, 'y_reg_test': y_reg_te,
    }

# Config A: daily Close return + polarity_mean
# Compute return series: (Close_t - Close_t-1) / Close_t-1
for split_df in [train_df, val_df, test_df]:
    split_df['return'] = split_df['Close'].pct_change().fillna(0)

CONFIG_A_FEATURES = ['return', 'polarity_mean']
data_A = build_data_variant(CONFIG_A_FEATURES, train_df, val_df, test_df)
print(f'Config A X_train: {data_A["X_train"].shape}  (2 features)')

# Config B: denoised OHLCV (5) + polarity_mean = 6 features
CONFIG_B_FEATURES = ['Open_d', 'High_d', 'Low_d', 'Close_d', 'Volume_d', 'polarity_mean']
data_B = build_data_variant(CONFIG_B_FEATURES, train_df, val_df, test_df)
print(f'Config B X_train: {data_B["X_train"].shape}  (6 features)')

print(f'Config C X_train: {data_C["X_train"].shape}  (8 features — already loaded)')

Split: Train=746 | Val=160 | Test=161
Split: Train=746 Val=160 Test=161
Config A X_train: (741, 5, 2)  (2 features)
Config B X_train: (741, 5, 6)  (6 features)
Config C X_train: (741, 5, 8)  (8 features — already loaded)


## Helper: Run One Config

In [4]:
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/kotekar_ablation_partial.csv'
all_results = pd.DataFrame()


def run_config(config_name, model_name, data, class_weights=None):
    """Run 30-seed training for one config and append to results CSV."""
    global all_results
    n_features = data['X_train'].shape[2]
    print(f'\n{"=" * 60}')
    print(f'Config {config_name} | {model_name.upper()} | n_features={n_features}')
    print(f'{"=" * 60}')

    results = train_multi_run(
        config_name=config_name,
        model_name=model_name,
        n_features=n_features,
        data=data,
        dataset='kotekar',
        class_weights=class_weights,
        device=device,
    )

    all_results = pd.concat([all_results, results], ignore_index=True)
    all_results.to_csv(RESULTS_PATH, index=False)
    print(f'Results saved ({len(all_results)} rows total)')

    mean_val = results['val_accuracy'].mean()
    mean_test = results['accuracy'].mean()
    print(f'Config {config_name} | mean val_acc={mean_val:.4f} | mean test_acc={mean_test:.4f}')

    torch.cuda.empty_cache()
    gc.collect()
    return results

## Tasks 3.2–3.4 — Run Configs A, B, C (TKAN, 30 seeds each)

In [5]:
results_A = run_config('A', 'tkan', data_A, class_weights)


Config A | TKAN | n_features=2
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 1/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 2/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 3/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 4/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 5/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 38 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 6/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 7/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config A | tkan | Run 8/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.60

In [6]:
results_B = run_config('B', 'tkan', data_B, class_weights)


Config B | TKAN | n_features=6
  Stopped at epoch 47 / 150 | Best val_acc: 0.6258
Config B | tkan | Run 1/30 | Val acc: 0.6258 | Test acc: 0.4359
  Stopped at epoch 49 / 150 | Best val_acc: 0.6194
Config B | tkan | Run 2/30 | Val acc: 0.6194 | Test acc: 0.4615
  Stopped at epoch 57 / 150 | Best val_acc: 0.6129
Config B | tkan | Run 3/30 | Val acc: 0.6129 | Test acc: 0.4679
  Stopped at epoch 56 / 150 | Best val_acc: 0.6194
Config B | tkan | Run 4/30 | Val acc: 0.6194 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config B | tkan | Run 5/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 46 / 150 | Best val_acc: 0.6129
Config B | tkan | Run 6/30 | Val acc: 0.6129 | Test acc: 0.4615
  Stopped at epoch 50 / 150 | Best val_acc: 0.6129
Config B | tkan | Run 7/30 | Val acc: 0.6129 | Test acc: 0.4231
  Stopped at epoch 49 / 150 | Best val_acc: 0.6129
Config B | tkan | Run 8/30 | Val acc: 0.6129 | Test acc: 0.4487
  Stopped at epoch 37 / 150 | Best val_acc: 0.60

In [7]:
results_C = run_config('C', 'tkan', data_C, class_weights)


Config C | TKAN | n_features=8
  Stopped at epoch 110 / 150 | Best val_acc: 0.7613
Config C | tkan | Run 1/30 | Val acc: 0.7613 | Test acc: 0.5513
  Stopped at epoch 133 / 150 | Best val_acc: 0.7484
Config C | tkan | Run 2/30 | Val acc: 0.7484 | Test acc: 0.5577
  Stopped at epoch 121 / 150 | Best val_acc: 0.7355
Config C | tkan | Run 3/30 | Val acc: 0.7355 | Test acc: 0.5385
  Stopped at epoch 91 / 150 | Best val_acc: 0.7419
Config C | tkan | Run 4/30 | Val acc: 0.7419 | Test acc: 0.5513
  Stopped at epoch 113 / 150 | Best val_acc: 0.7548
Config C | tkan | Run 5/30 | Val acc: 0.7548 | Test acc: 0.5256
  Stopped at epoch 84 / 150 | Best val_acc: 0.7290
Config C | tkan | Run 6/30 | Val acc: 0.7290 | Test acc: 0.5128
  Stopped at epoch 150 / 150 | Best val_acc: 0.7742
Config C | tkan | Run 7/30 | Val acc: 0.7742 | Test acc: 0.5897
  Stopped at epoch 87 / 150 | Best val_acc: 0.7484
Config C | tkan | Run 8/30 | Val acc: 0.7484 | Test acc: 0.5321
  Stopped at epoch 78 / 150 | Best val_acc:

## Task 3.5 — Compare B vs C → Set BEST_REPR

> **ACTION REQUIRED:** After running this cell, manually update `BEST_REPR` in `config/config.py` and push to GitHub before running D/E/F.

In [8]:
mean_val_B = results_B['val_accuracy'].mean()
mean_val_C = results_C['val_accuracy'].mean()

print('=' * 50)
print('B vs C Comparison (mean val accuracy, 30 seeds)')
print('=' * 50)
print(f'Config B (denoised OHLCV):         {mean_val_B:.4f}')
print(f'Config C (denoised technicals):    {mean_val_C:.4f}')

winner = 'denoised_ohlcv' if mean_val_B > mean_val_C else 'denoised_technicals'
winner_data = data_B if mean_val_B > mean_val_C else data_C
winner_label = 'B' if mean_val_B > mean_val_C else 'C'

print(f'\nWINNER: Config {winner_label} -> BEST_REPR = "{winner}"')
print()
print('ACTION: Update config/config.py:')
print(f'    \'BEST_REPR\': \'{winner}\',')
print('Then push to GitHub and re-clone before running D/E/F.')

B vs C Comparison (mean val accuracy, 30 seeds)
Config B (denoised OHLCV):         0.6062
Config C (denoised technicals):    0.7538

WINNER: Config C -> BEST_REPR = "denoised_technicals"

ACTION: Update config/config.py:
    'BEST_REPR': 'denoised_technicals',
Then push to GitHub and re-clone before running D/E/F.


## Task 3.6 — Reload Updated Config After BEST_REPR is Set

> Re-clone from GitHub after updating config.py and rerun this cell before D/E/F.

In [9]:
# After updating BEST_REPR in config.py and pushing to GitHub:
# subprocess.run(['git', '-C', '/kaggle/working/WavSent-MTL', 'pull'], check=True)

import importlib
import config.config as _cfg_mod
importlib.reload(_cfg_mod)
from config.config import CONFIG

BEST_REPR = CONFIG['BEST_REPR']
print(f'BEST_REPR = "{BEST_REPR}"')

# Select data for BEST_REPR
if BEST_REPR == 'denoised_ohlcv':
    data_BEST = data_B
elif BEST_REPR == 'denoised_technicals':
    data_BEST = data_C
else:
    raise ValueError(f'Unknown BEST_REPR: {BEST_REPR}')

print(f'Using data_BEST: X_train shape = {data_BEST["X_train"].shape}')

BEST_REPR = "denoised_technicals"
Using data_BEST: X_train shape = (741, 5, 8)


## Tasks 3.7–3.9 — Run Configs D (LSTM), E (GRU), F (TCN)

In [10]:
results_D = run_config('D', 'lstm', data_BEST, class_weights)


Config D | LSTM | n_features=8
  Stopped at epoch 37 / 150 | Best val_acc: 0.6000
Config D | lstm | Run 1/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config D | lstm | Run 2/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 36 / 150 | Best val_acc: 0.6000
Config D | lstm | Run 3/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 38 / 150 | Best val_acc: 0.6000
Config D | lstm | Run 4/30 | Val acc: 0.6000 | Test acc: 0.5641
  Stopped at epoch 83 / 150 | Best val_acc: 0.7742
Config D | lstm | Run 5/30 | Val acc: 0.7742 | Test acc: 0.6282
  Stopped at epoch 122 / 150 | Best val_acc: 0.7806
Config D | lstm | Run 6/30 | Val acc: 0.7806 | Test acc: 0.5833
  Stopped at epoch 100 / 150 | Best val_acc: 0.7742
Config D | lstm | Run 7/30 | Val acc: 0.7742 | Test acc: 0.6026
  Stopped at epoch 71 / 150 | Best val_acc: 0.7806
Config D | lstm | Run 8/30 | Val acc: 0.7806 | Test acc: 0.6026
  Stopped at epoch 36 / 150 | Best val_acc: 0.

In [11]:
results_E = run_config('E', 'gru', data_BEST, class_weights)


Config E | GRU | n_features=8
  Stopped at epoch 78 / 150 | Best val_acc: 0.7806
Config E | gru | Run 1/30 | Val acc: 0.7806 | Test acc: 0.6474
  Stopped at epoch 89 / 150 | Best val_acc: 0.7935
Config E | gru | Run 2/30 | Val acc: 0.7935 | Test acc: 0.5833
  Stopped at epoch 79 / 150 | Best val_acc: 0.8065
Config E | gru | Run 3/30 | Val acc: 0.8065 | Test acc: 0.6026
  Stopped at epoch 75 / 150 | Best val_acc: 0.7871
Config E | gru | Run 4/30 | Val acc: 0.7871 | Test acc: 0.6026
  Stopped at epoch 118 / 150 | Best val_acc: 0.7935
Config E | gru | Run 5/30 | Val acc: 0.7935 | Test acc: 0.6026
  Stopped at epoch 85 / 150 | Best val_acc: 0.8129
Config E | gru | Run 6/30 | Val acc: 0.8129 | Test acc: 0.5962
  Stopped at epoch 90 / 150 | Best val_acc: 0.8000
Config E | gru | Run 7/30 | Val acc: 0.8000 | Test acc: 0.5962
  Stopped at epoch 94 / 150 | Best val_acc: 0.7871
Config E | gru | Run 8/30 | Val acc: 0.7871 | Test acc: 0.5769
  Stopped at epoch 84 / 150 | Best val_acc: 0.7871
Confi

In [12]:
results_F = run_config('F', 'tcn', data_BEST, class_weights)


Config F | TCN | n_features=8
  Stopped at epoch 74 / 150 | Best val_acc: 0.8323
Config F | tcn | Run 1/30 | Val acc: 0.8323 | Test acc: 0.6090
  Stopped at epoch 93 / 150 | Best val_acc: 0.8258
Config F | tcn | Run 2/30 | Val acc: 0.8258 | Test acc: 0.5641
  Stopped at epoch 82 / 150 | Best val_acc: 0.8258
Config F | tcn | Run 3/30 | Val acc: 0.8258 | Test acc: 0.5833
  Stopped at epoch 101 / 150 | Best val_acc: 0.8323
Config F | tcn | Run 4/30 | Val acc: 0.8323 | Test acc: 0.5513
  Stopped at epoch 106 / 150 | Best val_acc: 0.8323
Config F | tcn | Run 5/30 | Val acc: 0.8323 | Test acc: 0.5962
  Stopped at epoch 98 / 150 | Best val_acc: 0.8258
Config F | tcn | Run 6/30 | Val acc: 0.8258 | Test acc: 0.5449
  Stopped at epoch 92 / 150 | Best val_acc: 0.8194
Config F | tcn | Run 7/30 | Val acc: 0.8194 | Test acc: 0.5577
  Stopped at epoch 78 / 150 | Best val_acc: 0.8194
Config F | tcn | Run 8/30 | Val acc: 0.8194 | Test acc: 0.6346
  Stopped at epoch 97 / 150 | Best val_acc: 0.8065
Conf

## Summary

In [13]:
# Save final complete results
final_path = '/kaggle/working/kotekar_ablation_AG.csv'
all_results.to_csv(final_path, index=False)

print('=' * 60)
print('Notebook 06 -- Kotekar Training: COMPLETE')
print('=' * 60)

summary = all_results.groupby('config').agg(
    mean_val_acc=('val_accuracy', 'mean'),
    mean_test_acc=('accuracy', 'mean'),
    std_test_acc=('accuracy', 'std'),
    mean_auc=('auc', 'mean'),
).round(4)
print(summary.to_string())

print(f'\nDownload these files from /kaggle/working/:')
print('  kotekar_ablation_AG.csv')
print('  kotekar_ablation_partial.csv  (incremental backup)')
print()
print('  kotekar_tkan_best.pt          ← best TKAN weights (for SHAP)')
print('  kotekar_lstm_best.pt          ← best LSTM weights (for SHAP)')
print('  kotekar_gru_best.pt           ← best GRU  weights (for SHAP)')
print('  kotekar_tcn_best.pt           ← best TCN  weights (for SHAP)')
print()
print('Val predictions saved at:')
print('  ablation/results/kotekar/val_predictions/*.npy')
print()
print('After downloading .pt files, place them at:')
print('  results/saved_models/kotekar/{model}_best.pt')
print()
print('Next: run notebook 07_training_kaggle')

Notebook 06 -- Kotekar Training: COMPLETE
        mean_val_acc  mean_test_acc  std_test_acc  mean_auc
config                                                     
A             0.6000         0.5641        0.0000    0.4993
B             0.6062         0.5162        0.0605    0.5314
C             0.7538         0.5442        0.0203    0.5906
D             0.6649         0.5724        0.0162    0.5407
E             0.7938         0.5921        0.0197    0.6576
F             0.8267         0.5744        0.0282    0.6423

Download these files from /kaggle/working/:
  kotekar_ablation_AG.csv
  kotekar_ablation_partial.csv  (incremental backup)

  kotekar_tkan_best.pt          ← best TKAN weights (for SHAP)
  kotekar_lstm_best.pt          ← best LSTM weights (for SHAP)
  kotekar_gru_best.pt           ← best GRU  weights (for SHAP)
  kotekar_tcn_best.pt           ← best TCN  weights (for SHAP)

Val predictions saved at:
  ablation/results/kotekar/val_predictions/*.npy

After downloading .pt fi